In [1]:
# Export gene expr for OF calculation

In [2]:
import scanpy as sc
adata = sc.read_h5ad("F4_VisiumHD.h5ad")  # full load
# or: adata = sc.read_h5ad("combinedHD_8um_v2.h5ad", backed="r")

In [3]:
# Export the observation metadata to a TSV file
adata.obs.to_csv("meta.tsv", sep="\t")

In [4]:
import pandas as pd

# 1. Extract the coordinates (e.g., 'X_umap' for UMAP, or change to 'spatial' if you want physical coordinates)
dim_red_key = 'X_umap' 
coords_array = adata.obsm[dim_red_key]

# 2. Convert to a DataFrame with matching cell/spot barcodes as the index
coords_df = pd.DataFrame(
    coords_array, 
    index=adata.obs_names, 
    columns=['UMAP1', 'UMAP2']  # Rename columns to match your specific reduction
)

# 3. Save as a compressed tab-separated file (.tsv.gz)
coords_df.to_csv("Visium.coords.tsv.gz", sep="\t", compression="gzip")

In [5]:
mapping = {"Assembloid": "A55", "VIO": "V10"}  # ensure '0' vs 'O' is correct
adata.obs["library_id"] = adata.obs["sample"].astype(str).map(mapping).astype("category")
adA55 = adata[adata.obs['library_id'] == 'A55'].copy()
adV10 = adata[adata.obs['library_id'] == 'V10'].copy()


In [6]:
# Convert to seurat for OFgene comparsion
import os
import pandas as pd
import scipy.sparse as sp
from scipy.io import mmwrite

# Output folder
outdir = "adA55_expression_only_for_seurat"
os.makedirs(outdir, exist_ok=True)

# Make names unique
adA55.var_names_make_unique()
adA55.obs_names_make_unique()

# Choose expression matrix
# Prefer raw counts if they exist
if "counts" in adA55.layers:
    X = adA55.layers["counts"]
    print("Using adA55.layers['counts']")
else:
    X = adA55.X
    print("Using adA55.X")

# Convert to sparse matrix if needed
if not sp.issparse(X):
    X = sp.csr_matrix(X)
else:
    X = X.tocsr()

# AnnData is cells x genes
# Seurat wants genes x cells, so transpose here
X_seurat = X.T

# Save matrix
mmwrite(os.path.join(outdir, "matrix.mtx"), X_seurat)

# Save gene names and cell names
pd.DataFrame(adA55.var_names).to_csv(
    os.path.join(outdir, "genes.tsv"),
    sep="\t",
    index=False,
    header=False
)

pd.DataFrame(adA55.obs_names).to_csv(
    os.path.join(outdir, "barcodes.tsv"),
    sep="\t",
    index=False,
    header=False
)

print("Done exporting expression matrix only.")
print(f"Genes: {adA55.n_vars}")
print(f"Cells/spots/bins: {adA55.n_obs}")

Using adA55.layers['counts']
Done exporting expression matrix only.
Genes: 18085
Cells/spots/bins: 63819
